In [1]:
## load and split data
import pandas as pd
from sklearn.model_selection import train_test_split
from data_cleaner import preprocess_features

df_pos = pd.read_csv('../../data/inputs/clf_descriptors_positive_staphylococcus.csv')
df_neg = pd.read_csv('../../data/inputs/clf_descriptors_negative_02_dataset.csv')

#concat
df = pd.concat([df_pos, df_neg])

#split - "ACTIVITY" column is the target
X = df.drop(columns=['SEQUENCE', 'ACTIVITY'])
y = df['ACTIVITY']
X = preprocess_features(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

-----Variance treshold-----
Features dropped: Index([], dtype='object')
New shape: (926, 102)

-----Correlation filter-----
Features dropped: ['F2', 'KF1', 'KF2', 'MSWHIM1', 'E1', 'PD1', 'PD2', 'PRIN1', 'PRIN2', 'ProtFP1', 'ProtFP2', 'ST1', 'SVGER7', 'SVGER10', 'T1', 'VHSE1', 'VHSE2', 'VHSE3', 'VSTPV1', 'Z1', 'Z2']
New shape: (926, 81)


In [2]:
## Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

param_grid = {
    'n_estimators': [450],
    'max_depth': [8],
    'criterion': ['entropy']
}
rf_grid = GridSearchCV(RandomForestClassifier(),
                       param_grid,
                       scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
                       refit="f1",
                       verbose=0,
                       return_train_score=True
).fit(X_train, y_train)
rf = rf_grid.best_estimator_

In [7]:
## SVM
from sklearn.svm import SVC

param_grid = {'C': [10],
              'gamma': [1],
              'kernel': ['rbf']
              }

svc_grid = GridSearchCV(
    SVC(),
    param_grid,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    refit="f1", # podle tohohle se vybere nejlepší model
    verbose=0,
    return_train_score=True
).fit(X_train, y_train)

svc_grid.fit(X_train, y_train)
svc = svc_grid.best_estimator_
svc_grid.cv_results_

{'mean_fit_time': array([0.01500797]),
 'std_fit_time': array([0.00337053]),
 'mean_score_time': array([0.02250304]),
 'std_score_time': array([0.00483476]),
 'param_C': masked_array(data=[10],
              mask=[False],
        fill_value=999999),
 'param_gamma': masked_array(data=[1],
              mask=[False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf'],
              mask=[False],
        fill_value=np.str_('?'),
             dtype=object),
 'params': [{'C': 10, 'gamma': 1, 'kernel': 'rbf'}],
 'split0_test_accuracy': array([0.87162162]),
 'split1_test_accuracy': array([0.90540541]),
 'split2_test_accuracy': array([0.89864865]),
 'split3_test_accuracy': array([0.85810811]),
 'split4_test_accuracy': array([0.84459459]),
 'mean_test_accuracy': array([0.87567568]),
 'std_test_accuracy': array([0.02324953]),
 'rank_test_accuracy': array([1], dtype=int32),
 'split0_train_accuracy': array([0.98986486]),
 'split1_train_accuracy': array([0.98648649]),
 'split2_t

In [4]:
## Naive Bayes
from sklearn.naive_bayes import GaussianNB

nb_grid = GridSearchCV(GaussianNB(),
                       {},
                        scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
                        refit="f1", # podle tohohle se vybere nejlepší model
                        verbose=0,
                        return_train_score=True)
nb_grid.fit(X_train, y_train)
nb = nb_grid.best_estimator_
nb_grid.cv_results_

{'mean_fit_time': array([0.0027998]),
 'std_fit_time': array([0.00075019]),
 'mean_score_time': array([0.00940847]),
 'std_score_time': array([0.00037473]),
 'params': [{}],
 'split0_test_accuracy': array([0.82432432]),
 'split1_test_accuracy': array([0.80405405]),
 'split2_test_accuracy': array([0.85135135]),
 'split3_test_accuracy': array([0.84459459]),
 'split4_test_accuracy': array([0.86486486]),
 'mean_test_accuracy': array([0.83783784]),
 'std_test_accuracy': array([0.02136674]),
 'rank_test_accuracy': array([1], dtype=int32),
 'split0_train_accuracy': array([0.84290541]),
 'split1_train_accuracy': array([0.84459459]),
 'split2_train_accuracy': array([0.83783784]),
 'split3_train_accuracy': array([0.84121622]),
 'split4_train_accuracy': array([0.83108108]),
 'mean_train_accuracy': array([0.83952703]),
 'std_train_accuracy': array([0.00477775]),
 'split0_test_precision': array([0.85074627]),
 'split1_test_precision': array([0.84375]),
 'split2_test_precision': array([0.91803279]),

In [5]:
# Neural Network
from neural_network_from_gemini import train_and_evaluate_mlp
# model, accuracy, report, confusion, scaler = train_and_evaluate_mlp(df.drop(columns=['SEQUENCE']))
# nn = model

ModuleNotFoundError: No module named 'neural_network_from_gemini'

## Visualization

In [ ]:
# ROC curve
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

ax = plt.gca()
disp = RocCurveDisplay.from_estimator(svc, X_test, y_test, ax=ax)
disp = RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=ax)
disp = RocCurveDisplay.from_estimator(nb, X_test, y_test, ax=ax)
plt.title('Receiver Operating Characteristic (ROC) Curves')

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

ax = plt.gca()
disp = PrecisionRecallDisplay.from_estimator(svc, X_test, y_test, ax=ax)
disp = PrecisionRecallDisplay.from_estimator(rf, X_test, y_test, ax=ax)
disp = PrecisionRecallDisplay.from_estimator(nb, X_test, y_test, ax=ax)

...used: https://www.datacamp.com/tutorial/python-boxplots

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

svm_values = np.concatenate([np.array([svc_grid.cv_results_[f'split{i}_test_accuracy'][0] for i in range(5)]), 
                            np.array([svc_grid.cv_results_[f'split{i}_test_precision'][0] for i in range(5)]),
                            np.array([svc_grid.cv_results_[f'split{i}_test_recall'][0] for i in range(5)]),
                            np.array([svc_grid.cv_results_[f'split{i}_test_roc_auc'][0] for i in range(5)])])
rf_values = np.concatenate([np.array([rf_grid.cv_results_[f'split{i}_test_accuracy'][0] for i in range(5)]),
                            np.array([rf_grid.cv_results_[f'split{i}_test_precision'][0] for i in range(5)]),
                            np.array([rf_grid.cv_results_[f'split{i}_test_recall'][0] for i in range(5)]),
                            np.array([rf_grid.cv_results_[f'split{i}_test_roc_auc'][0] for i in range(5)])])
nb_values = np.concatenate([np.array([nb_grid.cv_results_[f'split{i}_test_accuracy'][0] for i in range(5)]),
                            np.array([nb_grid.cv_results_[f'split{i}_test_precision'][0] for i in range(5)]),
                            np.array([nb_grid.cv_results_[f'split{i}_test_recall'][0] for i in range(5)]),
                            np.array([nb_grid.cv_results_[f'split{i}_test_roc_auc'][0] for i in range(5)])])
means = np.array([np.array(svc_grid.cv_results_['mean_test_precision'][0]),
                            np.array(svc_grid.cv_results_['mean_test_accuracy'][0]),
                            np.array(svc_grid.cv_results_['mean_test_recall'][0]),
                            np.array(svc_grid.cv_results_['mean_test_roc_auc'][0]),
                            np.array(rf_grid.cv_results_['mean_test_precision'][0]),
                            np.array(rf_grid.cv_results_['mean_test_accuracy'][0]),
                            np.array(rf_grid.cv_results_['mean_test_recall'][0]),
                            np.array(rf_grid.cv_results_['mean_test_roc_auc'][0]),
                            np.array(nb_grid.cv_results_['mean_test_precision'][0]),
                            np.array(nb_grid.cv_results_['mean_test_accuracy'][0]),
                            np.array(nb_grid.cv_results_['mean_test_recall'][0]),
                            np.array(nb_grid.cv_results_['mean_test_roc_auc'][0])])
                            
std_devs = np.array([np.array(svc_grid.cv_results_['std_test_precision'][0]),
                           np.array(svc_grid.cv_results_['std_test_accuracy'][0]),
                            np.array(svc_grid.cv_results_['std_test_recall'][0]),
                            np.array(svc_grid.cv_results_['std_test_roc_auc'][0]),
                            np.array(rf_grid.cv_results_['std_test_precision'][0]),
                            np.array(rf_grid.cv_results_['std_test_accuracy'][0]),
                            np.array(rf_grid.cv_results_['std_test_recall'][0]),
                            np.array(rf_grid.cv_results_['std_test_roc_auc'][0]),
                            np.array(nb_grid.cv_results_['std_test_precision'][0]),
                            np.array(nb_grid.cv_results_['std_test_accuracy'][0]),
                            np.array(nb_grid.cv_results_['std_test_recall'][0]),
                            np.array(nb_grid.cv_results_['std_test_roc_auc'][0])])

print(std_devs)


data = {
    'Model': ['SVM']*20 + ['RF']*20 + ['NB']*20,
    'Metric': (['Accuracy']*5 + ['Precision']*5 + ['Recall']*5 + ['ROC_AUC']*5) * 3,
    'Value': np.concatenate([svm_values, rf_values, nb_values])
}

df = pd.DataFrame(data)
# Create the boxplot
sns.boxplot(x='Model', y='Value', data=df, hue='Metric', palette='Set2')
plt.title('Comparison of performance metrics for all models.')
plt.show()

In [ ]:
# SVM plot
svm_accuracy = np.array([svc_grid.cv_results_[f'split{i}_test_accuracy'][0] for i in range(5)])
svm_precision = np.array([svc_grid.cv_results_[f'split{i}_test_precision'][0] for i in range(5)])
svm_recall = np.array([svc_grid.cv_results_[f'split{i}_test_recall'][0] for i in range(5)])
svm_roc_auc = np.array([svc_grid.cv_results_[f'split{i}_test_roc_auc'][0] for i in range(5)])

svm_std_devs = np.array([np.array(svc_grid.cv_results_['std_test_accuracy'][0]),
                            np.array(svc_grid.cv_results_['std_test_precision'][0]),
                             np.array(svc_grid.cv_results_['std_test_recall'][0]),
                             np.array(svc_grid.cv_results_['std_test_roc_auc'][0])])
svm_means = np.array([np.array(svc_grid.cv_results_['mean_test_accuracy'][0]),
                            np.array(svc_grid.cv_results_['mean_test_precision'][0]),
                            np.array(svc_grid.cv_results_['mean_test_recall'][0]),
                            np.array(svc_grid.cv_results_['mean_test_roc_auc'][0])])
data = svm_accuracy, svm_precision, svm_recall, svm_roc_auc
plt.boxplot(data, tick_labels=['Accuracy', 'Precision', 'Recall', 'ROC_AUC'])
# Adds mean as red dots
for i in range(len(svm_means)):
   plt.plot(i + 1, svm_means[i], 'ro')
# Adds standard deviations as error bars
for i in range(len(svm_std_devs)):
   plt.errorbar(i + 1, svm_means[i], yerr=svm_std_devs[i], fmt='o', color='red')
# Plots graph
plt.title('SVM performance metrics')
ax = plt.gca()
ax.set_ylim([0.7, 1])
plt.xlabel('Dataset')
plt.ylabel('Value')
plt.show()

In [ ]:
# RF plot
rf_accuracy = np.array([rf_grid.cv_results_[f'split{i}_test_accuracy'][0] for i in range(5)])
rf_precision = np.array([rf_grid.cv_results_[f'split{i}_test_precision'][0] for i in range(5)])
rf_recall = np.array([rf_grid.cv_results_[f'split{i}_test_recall'][0] for i in range(5)])
rf_roc_auc = np.array([rf_grid.cv_results_[f'split{i}_test_roc_auc'][0] for i in range(5)])

rf_std_devs = np.array([np.array(rf_grid.cv_results_['std_test_accuracy'][0]),
                            np.array(rf_grid.cv_results_['std_test_precision'][0]),
                             np.array(rf_grid.cv_results_['std_test_recall'][0]),
                             np.array(rf_grid.cv_results_['std_test_roc_auc'][0])])
rf_means = np.array([np.array(rf_grid.cv_results_['mean_test_accuracy'][0]),
                            np.array(rf_grid.cv_results_['mean_test_precision'][0]),
                            np.array(rf_grid.cv_results_['mean_test_recall'][0]),
                            np.array(rf_grid.cv_results_['mean_test_roc_auc'][0])])
data = rf_accuracy, rf_precision, rf_recall, rf_roc_auc
plt.boxplot(data, tick_labels=['Accuracy', 'Precision', 'Recall', 'ROC_AUC'])
# Adds mean as red dots
for i in range(len(rf_means)):
   plt.plot(i + 1, rf_means[i], 'ro')
# Adds standard deviations as error bars
for i in range(len(rf_std_devs)):
   plt.errorbar(i + 1, rf_means[i], yerr=rf_std_devs[i], fmt='o', color='red')
# Plots graph
plt.title('RF performance metrics')
ax = plt.gca()
ax.set_ylim([0.7, 1])
plt.xlabel('Dataset')
plt.ylabel('Value')
plt.show()

In [ ]:
# NB plot
nb_accuracy = np.array([nb_grid.cv_results_[f'split{i}_test_accuracy'][0] for i in range(5)])
nb_precision = np.array([nb_grid.cv_results_[f'split{i}_test_precision'][0] for i in range(5)])
nb_recall = np.array([nb_grid.cv_results_[f'split{i}_test_recall'][0] for i in range(5)])
nb_roc_auc = np.array([nb_grid.cv_results_[f'split{i}_test_roc_auc'][0] for i in range(5)])

nb_std_devs = np.array([np.array(nb_grid.cv_results_['std_test_accuracy'][0]),
                            np.array(nb_grid.cv_results_['std_test_precision'][0]),
                             np.array(nb_grid.cv_results_['std_test_recall'][0]),
                             np.array(nb_grid.cv_results_['std_test_roc_auc'][0])])
nb_means = np.array([np.array(nb_grid.cv_results_['mean_test_accuracy'][0]),
                            np.array(nb_grid.cv_results_['mean_test_precision'][0]),
                            np.array(nb_grid.cv_results_['mean_test_recall'][0]),
                            np.array(nb_grid.cv_results_['mean_test_roc_auc'][0])])
data = nb_accuracy, nb_precision, nb_recall, nb_roc_auc
plt.boxplot(data, tick_labels=['Accuracy', 'Precision', 'Recall', 'ROC_AUC'])
# Adds mean as red dots
for i in range(len(nb_means)):
   plt.plot(i + 1, nb_means[i], 'ro')
# Adds standard deviations as error bars
for i in range(len(nb_std_devs)):
   plt.errorbar(i + 1, nb_means[i], yerr=nb_std_devs[i], fmt='o', color='red')
# Plots graph
plt.title('NB performance metrics')
ax = plt.gca()
ax.set_ylim([0.7, 1])
plt.xlabel('Dataset')
plt.ylabel('Value')
plt.show()